# Frequency-domain FWI with `odil_wave.experiment`


## Imports & repo paths


In [ ]:
# Force inline (interactive) figures so plt.show() renders in the notebook.
%matplotlib inline

import json
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

import odil_wave
from odil_wave.experiment import (
    load_config_file, deep_merge, resolve_config,
    build_problem, run_inverse,
    LiveVelocityView, plot_run_history, plot_velocity_recovery, animate_bands,
    load_metrics, load_loss_tape,
)
from odil_wave.metrics import ssim, ssim_map

ODIL_ROOT = Path(odil_wave.__file__).resolve().parents[1]
OUTPUT_ROOT = ODIL_ROOT / "outputs" / "experiment_runs"  # git-ignored
DEFAULT_CONFIG = ODIL_ROOT / "configs" / "default_inverse.yaml"
assert DEFAULT_CONFIG.exists(), DEFAULT_CONFIG
print("odil root      :", ODIL_ROOT)
print("default config :", DEFAULT_CONFIG.relative_to(ODIL_ROOT))

## 1. Choose a configuration

In [ ]:
base_cfg = load_config_file(DEFAULT_CONFIG)

OVERDENSITY_BASE = 1500.0
OVERDENSITY_CONTRAST = 400.0
OVERDENSITY_RADIUS = 0.05

# Short run (~2 min on CPU): 32 of the 64 ring sources and a two-band 40 -> 60 kHz
# continuation, 20 joint L-BFGS outer steps per band.
FULL_CONFIG = {
    "run": {"output_root": str(OUTPUT_ROOT)},
    "grid": {"interior_shape": [125, 125], "init_nt": 1300, "pml_width": 40},
    "init": {"profile": "homogeneous"},
    "truth": {},
    "acquisition": {"n_sources": 32},
    "optimiser": {
        "name": "joint",
        "n_iter": 20,
        "joint": {
            "inner_max_iter": 20,
            "history_size": 20,
            "data_weight": 1.0,
        }
    },
    "physics": {"space_order": 4},
    "continuation": {
        "bands": [
            {"frequencies_hz": [40e3]},
            {"frequencies_hz": [60e3]},
        ]
    },
    # loss.weights / regulariser feed the logged loss decomposition; the joint
    # objective uses optimiser.joint.data_weight / reg_weight instead.
    "loss": {
        "weights": {"pde": 1.0, "data": 1000.0, "reg": 5.0e-05},
        "regulariser": {"name": "tv_iso"}
    },
    "observation": {
        "normalize_data": None
    }
}

user_cfg = deep_merge(base_cfg, FULL_CONFIG)    # replace config fields

cfg = resolve_config(user_cfg)
print("run_id       :", cfg.run.run_id)
print("optimiser    :", cfg.optimiser.name)
print("truth        :", cfg.truth.profile,
      f"| base {cfg.truth.base:.0f} + contrast {cfg.truth.contrast:.0f} m/s"
      f" | radius {cfg.truth.extra.get('radius')} m")
print("grid interior:", cfg.grid.interior_shape, "| pml:", cfg.grid.pml_width)
print("observation  : helmholtz on truth | normalize:", cfg.observation.normalize_data,
      "| pml:", cfg.observation.pml_width)
print("bands [kHz]  :", [[f / 1e3 for f in b.frequencies_hz] for b in cfg.continuation.bands])

## 2. Build the problem and reproduce the setup diagnostics

`build_problem(cfg)` builds the grid, source, acquisition layout and truth /
initial models once — the same object every solver operates on. Per-band pieces
(frequency selection, geometry, observed wavefields) come from
`problem.make_band(...)`.


In [ ]:
problem = build_problem(cfg)
grid = problem.grid
print(grid.summary)

band0 = problem.make_band(cfg.continuation.bands[0].frequencies_hz)
print("selected frequencies [kHz]:", (band0.freq.frequencies / 1e3).tolist())
print("FFT bins                  :", band0.freq.fft_bins.tolist())

grid.plot_absorption_profile(); plt.show()
band0.geom.show(problem.truth_velocity)
band0.geom.show(problem.init_velocity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
problem.source.plot_pulse(ax=axes[0])
band0.geom.plot_source_field(src_idx=0, ax=axes[1])
plt.tight_layout(); plt.show()

# Source magnitude spectrum with the inversion bins marked (library helper).
problem.source.plot_spectrum(frequencies=band0.freq)
plt.show()

In [ ]:
# Observed (truth) wavefield on the dominant bin. Wavefield.show now draws a
# 2x2 of |u|, Re(u), Im(u) and phase in one call.
k = int(np.argmin(np.abs(band0.freq.frequencies.abs().cpu().numpy() - problem.source.f0)))
band0.observed_wfs[0].show(k, title="Observed shot 0 (truth)")

## 3. Run the inversion — one call, one artifact directory

`run_inverse` builds the problem, runs every continuation band with the selected
optimiser, logs the metric tape, and saves per-band + final velocity fields.
Passing `input_config` also preserves the raw user config to
`config_input.yaml`.


In [ ]:
live = LiveVelocityView(problem.truth_velocity)
result = run_inverse(cfg, input_config=user_cfg, on_band_end=live)
run_dir = result.run_dir

print("status  :", result.status)
print("run_dir :", run_dir)
print("summary :", json.dumps(result.metrics_summary, indent=2))

## 4. The per-iteration metric tape

`metrics.jsonl` / `metrics.csv` hold one scalar record per logged iteration.
`plot_run_history` draws the per-band loss & SSIM directly from them, and
`load_loss_tape` rebuilds a `LossTape` so its own `show()` works offline.

In [ ]:
# Per-band loss & head-ROI SSIM straight from the saved metrics.jsonl.
records = load_metrics(run_dir)
print(f"{len(records)} logged iterations; columns:")
print(list(records[0].keys()))
plot_run_history(run_dir, title="Loss & head-ROI SSIM per band")

# The saved scalars also reconstruct a LossTape, so its built-in plots work
# fully offline (loss / PDE-RMS / data-RMS curves).
tape = load_loss_tape(run_dir)
tape.show(title="Loss / PDE / data RMS (reconstructed from disk)")

## 5. Velocity recovery — from the saved artifacts

`plot_velocity_recovery` loads `final/c_final.npy` and the `rel_c_error` series
to draw truth / recovered / difference / recovery-error; `ssim_map` shows the
per-pixel SSIM inside the head ROI; `animate_bands` renders the continuation
sweep (one frame per band).

In [ ]:
# 4-panel truth | recovered | difference | recovery-error, from saved
# artifacts (final/c_final.npy + the rel_c_error series in metrics.jsonl).
plot_velocity_recovery(run_dir, problem.truth_velocity, band0.geom,
                       title="Velocity recovery")

# Per-pixel SSIM map inside the head ROI, recomputed from disk.
c_final = np.load(run_dir / "final" / "c_final.npy")
c_final_t = torch.from_numpy(c_final).to(problem.truth_velocity.c)
s = ssim(c_final_t, problem.truth_velocity.c, grid, mask=problem.head_mask)
print(f"final head-ROI SSIM (recomputed from disk): {s:.4f}")
ssim_map(c_final_t, problem.truth_velocity.c, grid,
         title=f"SSIM map (SSIM={s:.3f})", mask=problem.head_mask)
plt.show()

# The frequency-continuation sweep as an animation (one frame per band).
gif = animate_bands(run_dir, grid, filename=str(run_dir / "c_bands.gif"), fps=1)

In [ ]:
# run_plot_dir = OUTPUT_ROOT / "<DDMMYY_HHMMSS>_shepp-logan_125x125_joint_80_40-60"

# plot_run_history(run_plot_dir, title="Loss & head-ROI SSIM per band")

# plot_velocity_recovery(run_plot_dir, title="Velocity recovery (from run dir alone)")

# slow_gif = animate_bands(
#     run_plot_dir,
#     filename=str(run_plot_dir / "c_bands_slow.gif"),
#     fps=1,
#     title="Recovered c across bands (slow)",
# )
# print("slow band animation:", slow_gif)

## 6. Warm-restart: seed a new run from a previous run's final model

In [ ]:
# # Point at the previous run whose recovered model we want to continue from.
# PREV_RUN = OUTPUT_ROOT / "<DDMMYY_HHMMSS>_shepp-logan_125x125_joint_80_40-60"
# prev_final = PREV_RUN / "final" / "c_final.npy"
# assert prev_final.exists(), prev_final

# warm_cfg_user = deep_merge(user_cfg, {
#     "init": {
#         "profile": "field",
#         "pml_c": 1500.0,                          # source run's PML fill (water)
#         "extra": {"path": str(prev_final)},
#     },
#     "continuation": {"bands": [
#         {"frequencies_hz": [140e3]},
#     ]},
# })
# warm_cfg = resolve_config(warm_cfg_user)
# print("warm run_id :", warm_cfg.run.run_id)
# print("higher bands:", [[f / 1e3 for f in b.frequencies_hz] for b in warm_cfg.continuation.bands])

# # Build the problem and confirm the starting model IS the previous run's
# # recovered field (not an analytic profile). Truth is unchanged, so the head-ROI
# # SSIM of the *starting* model is already the low-band recovery quality.
# warm_problem = build_problem(warm_cfg)
# c_prev = np.load(prev_final)
# assert warm_problem.init_velocity.c.shape == warm_problem.grid.shape
# assert np.allclose(warm_problem.init_velocity.c.cpu().numpy(), c_prev, atol=1e-3)
# s0 = ssim(warm_problem.init_velocity.c, warm_problem.truth_velocity.c,
#           warm_problem.grid, mask=warm_problem.head_mask)
# print(f"starting model head-ROI SSIM (previous run's final): {s0:.4f}")
# warm_problem.init_velocity.show(title="Warm-start initial model = previous run's final c")
# plt.show()

In [ ]:
# warm_live = LiveVelocityView(warm_problem.truth_velocity)
# warm_result = run_inverse(warm_cfg, input_config=warm_cfg_user, on_band_end=warm_live)

# print("status  :", warm_result.status)
# print("run_dir :", warm_result.run_dir)
# print("summary :", json.dumps(warm_result.metrics_summary, indent=2))

# # Recovery vs truth, using the warm problem's own band geometry for the plot.
# warm_band0 = warm_problem.make_band(warm_cfg.continuation.bands[0].frequencies_hz)
# plot_run_history(warm_result.run_dir, title="Warm-restart: loss & head-ROI SSIM (higher bands)")
# plot_velocity_recovery(warm_result.run_dir, warm_problem.truth_velocity, warm_band0.geom,
#                        title="Warm-restart velocity recovery")